In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.indexes import create_vector_index

URI = "neo4j://10.0.40.100:7687"
AUTH = ("neo4j", "sacf_sacf")

INDEX_NAME = "probill"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)

# Creating the index
create_vector_index(
    driver,
    INDEX_NAME,
    label="Document",
    embedding_property="vectorProperty",
    dimensions=1536,
    similarity_fn="euclidean",
)

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.embeddings.ollama import OllamaEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

INDEX_NAME = "vector-index-name"

# Connect to Neo4j database
driver = GraphDatabase.driver(URI, auth=AUTH)
# Create Embedder object
# Note: An OPENAI_API_KEY environment variable is required here
embedder = OllamaEmbeddings(model="nomic-embed-text", host="http://10.0.40.49:11434")

# Initialize the retriever
retriever = VectorRetriever(driver, INDEX_NAME, embedder)

# Run the similarity search
query_text = "How do I do similarity search in Neo4j?"
response = retriever.search(query_text=query_text, top_k=5)


In [ ]:
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import SseMcpToolAdapter, mcp_server_tools,StdioServerParams
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_core import CancellationToken
from pathlib import Path

mcp_server_path = str("/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mcp_tools/mcp-neo4j/servers/mcp-neo4j-cypher")
server_params = StdioServerParams(
    command="uv", args=[
        "--directory",
        mcp_server_path,
        "run",
        "mcp-neo4j-cypher",
        "--db-url",
        "bolt://10.0.40.49:7687",
        "--username", "neo4j",
        "--password",
        "sacf_sacf"
    ]
)

tools = await mcp_server_tools(server_params)

In [ ]:
import json
tools_json = []
for tool in tools:
    # print(tool.name)
    tool_json = tool.dump_component().model_dump()
    tool_json["label"] = f"ne4j.{tool.name}"
    tool_json["config"]["name"] = f"ne4j.{tool.name}"
    tool_json["config"]["description"] = tool_json["description"]
    tools_json.append(tool_json)

print(json.dumps(tools_json, indent=2))

In [ ]:
import json
temp_json = {
  "nodes": [
    {
      "identifier": "Patient",
      "labels": ["Person"],
      "properties": {
        "age": 67,
        "sex": "Male"
      }
    },
    {
      "identifier": "Disease",
      "labels": ["LungCancer"],
      "properties": {
        "type": "NSCLC",
        "stage": "Stage IIIA"
      }
    },
    {
      "identifier": "Symptom",
      "labels": ["Symptom"],
      "properties": {
        "name": "Cough"
      }
    },
    {
      "identifier": "Biopsy",
      "labels": ["Biopsy"],
      "properties": {
        "result": "NSCLC Adenocarcinoma"
      }
    },
    {
      "identifier": "Mutation",
      "labels": ["Mutation"],
      "properties": {
        "type": "EGFR",
        "status": "Negative"
      }
    },
    {
      "identifier": "Treatment",
      "labels": ["Chemotherapy"],
      "properties": {
        "name": "Cisplatin + Pemetrexed"
      }
    },
    {
      "identifier": "DiagnosticTest",
      "labels": ["DiagnosticTest"],
      "properties": {
        "name": "CT Scan",
        "finding": "Right upper lobe mass"
      }
    }
  ],
  "relationships": [
    {
      "startNode": "Patient",
      "type": "HAS_CONDITION",
      "endNode": "Disease"
    },
    {
      "startNode": "Disease",
      "type": "PRESENTS_WITH",
      "endNode": "Symptom"
    },
    {
      "startNode": "Disease",
      "type": "DIAGNOSED_BY",
      "endNode": "Biopsy"
    },
    {
      "startNode": "Biopsy",
      "type": "HAS_MUTATION",
      "endNode": "Mutation"
    },
    {
      "startNode": "Disease",
      "type": "TREATED_WITH",
      "endNode": "Treatment"
    },
    {
      "startNode": "Patient",
      "type": "UNDERGOES",
      "endNode": "DiagnosticTest"
    }
  ]
}
print(temp_json)

In [ ]:
import xml.etree.ElementTree as ET
import re
from typing import List, Dict, Set, Tuple

class DecisionRule:
    """
    Represents a single DMN rule with:
      - a list of (index, text) pairs for input conditions (the i-th condition belongs to i-th input)
      - a set of sanitized outcomes
      - a textual description
    """
    def __init__(self, rule_id: str, description: str):
        self.id = rule_id
        self.description = description
        # Store input conditions as (clauseIndex, rawText).
        # We'll link clauseIndex -> the i-th input label in the decisionTable.
        self.conditions: List[Tuple[int, str]] = []
        self.outcomes: Set[str] = set()

class DecisionNode:
    """
    Represents a DMN decision (with a single decisionTable) plus:
      - an ordered list of input labels (the i-th label -> i-th inputEntry in each rule)
      - an ordered list of output names (rarely needed but consistent with DMN)
      - a list of rules
    """
    def __init__(self, decision_id: str, name: str):
        self.id = decision_id
        self.name = name
        # We’ll store input labels in an *ordered* list to match them by index
        self.ordered_inputs: List[str] = []
        # Also keep a set of inputs for references (some DMN files have <dmn:input label="...">)
        self.inputs_set: Set[str] = set()

        # Output labels likewise
        self.ordered_outputs: List[str] = []
        self.outputs_set: Set[str] = set()

        # IDs of other decisions required
        self.requires: Set[str] = set()
        # The DMN rules
        self.rules: List[DecisionRule] = []

class DecisionGraph:
    def __init__(self):
        self.decisions: Dict[str, DecisionNode] = {}
        self.guideline_name = ""
        self.guideline_version = ""

    def sanitize(self, text: str) -> str:
        """Sanitize text for Cypher compatibility by removing invalid characters."""
        if text is None:
            return "Unknown"
        return re.sub(r'[^a-zA-Z0-9_]', '_', text)

    def parse_dmn(self, dmn_files: List[str]):
        """
        Parses DMN files, capturing the ordered inputs (the i-th <input>),
        the rules (<rule>), and the i-th <inputEntry> for each rule, etc.
        """
        NS = {'dmn': 'https://www.omg.org/spec/DMN/20191111/MODEL/'}

        for file_path in dmn_files:
            tree = ET.parse(file_path)
            root = tree.getroot()

            for decision_el in root.findall('dmn:decision', NS):
                decision_id = decision_el.get('id') or "UnnamedDecision"
                decision_name = decision_el.get('name') or "Unnamed Decision"

                # Create or reuse DecisionNode
                if decision_id not in self.decisions:
                    self.decisions[decision_id] = DecisionNode(decision_id, decision_name)
                node = self.decisions[decision_id]

                # Check for required decisions
                for req_decision in decision_el.findall('.//dmn:requiredDecision', NS):
                    href = req_decision.get('href') or ""
                    node.requires.add(href.replace('#', ''))

                # We assume there's a single <decisionTable> per <decision>
                dtable = decision_el.find('dmn:decisionTable', NS)
                if dtable is None:
                    # Some DMN might have literal expressions or other forms. Skip if no table?
                    continue

                # 1) Collect the <input> elements in order
                #    We'll store their labels in node.ordered_inputs
                inputs = dtable.findall('dmn:input', NS)
                node.ordered_inputs = []
                for inp_el in inputs:
                    label = inp_el.get('label') or ""
                    node.ordered_inputs.append(label)
                    node.inputs_set.add(label)

                # 2) Collect the <output> elements in order
                outputs = dtable.findall('dmn:output', NS)
                node.ordered_outputs = []
                for out_el in outputs:
                    label = out_el.get('label') or out_el.get('name') or ""
                    node.ordered_outputs.append(label)
                    node.outputs_set.add(label)

                # 3) For each <rule>, gather i-th <inputEntry> => the i-th input's condition
                for rule_el in dtable.findall('dmn:rule', NS):
                    rule_id = rule_el.get('id') or "UnnamedRule"
                    desc_el = rule_el.find('.//dmn:annotation', NS)
                    description_text = desc_el.text if desc_el is not None else ""
                    rule_obj = DecisionRule(rule_id, description_text)

                    # Collect the inputEntry in the same index as the <input>
                    input_entries = rule_el.findall('dmn:inputEntry', NS)
                    for idx, in_entry in enumerate(input_entries):
                        cond_text_el = in_entry.find('dmn:text', NS)
                        cond_text = cond_text_el.text if (cond_text_el is not None) else ""
                        rule_obj.conditions.append((idx, cond_text))

                    # Collect the outputEntry as outcomes
                    output_entries = rule_el.findall('dmn:outputEntry', NS)
                    for out_entry in output_entries:
                        text_el = out_entry.find('dmn:text', NS)
                        raw_out = text_el.text if (text_el is not None) else "Unknown"
                        rule_obj.outcomes.add(self.sanitize(raw_out))

                    node.rules.append(rule_obj)

    def parse_bpmn(self, bpmn_file: str):
        """
        Parse BPMN to get the guideline name/version.
        """
        namespace = {'bpmn': 'http://www.omg.org/spec/BPMN/20100524/MODEL'}
        tree = ET.parse(bpmn_file)
        root = tree.getroot()

        process = root.find(".//bpmn:process[@isExecutable='true']", namespace)
        if process is not None:
            guideline_name_version = process.get('name') or "UnknownGuideline"
            parts = guideline_name_version.rsplit(' ', 1)
            self.guideline_name = parts[0]
            if len(parts) > 1:
                self.guideline_version = parts[1]
            else:
                self.guideline_version = "1.0"
        else:
            self.guideline_name = "UnknownGuideline"
            self.guideline_version = "1.0"

    def generate_cypher(self) -> str:
        """
        Generate Cypher MERGE statements for:
          - A single Guideline node
          - One node per Decision
          - One node per Input / Output (unique variable names via counters)
          - One node per Rule
          - One node per Condition, linked to the correct Input via :CONDITION_ON
          - One node per Outcome
        """
        statements = []
        statements.append(
            f"MERGE (g:Guideline {{name: '{self.guideline_name}', version: '{self.guideline_version}'}})"
        )

        # Keep global counters for unique variable names
        decision_idx = 0
        input_idx = 0
        output_idx = 0
        rule_idx = 0
        cond_idx = 0
        outcome_idx = 0

        for decision_id, node in self.decisions.items():
            # Decision
            d_var = f"d_{decision_idx}"
            decision_idx += 1
            d_merge = f"MERGE ({d_var}:Decision {{name: '{node.name}'}})"
            statements.append(d_merge)
            statements.append(f"MERGE (g)-[:HAS_DECISION]->({d_var})")

            # Inputs
            # We'll keep track of the variable name for the i-th input
            input_vars = []
            for i, inp_label in enumerate(node.ordered_inputs):
                inp_var = f"inp_{input_idx}"
                input_idx += 1
                label_sanitized = self.sanitize(inp_label)
                statements.append(
                    f"MERGE ({inp_var}:Input {{name: '{label_sanitized}'}})"
                )
                statements.append(
                    f"MERGE ({d_var})-[:NEEDS_INPUT]->({inp_var})"
                )
                input_vars.append(inp_var)

            # Outputs
            output_vars = []
            for i, out_label in enumerate(node.ordered_outputs):
                out_var = f"out_{output_idx}"
                output_idx += 1
                out_sanitized = self.sanitize(out_label)
                statements.append(
                    f"MERGE ({out_var}:Output {{name: '{out_sanitized}'}})"
                )
                statements.append(
                    f"MERGE ({d_var})-[:HAS_OUTPUT]->({out_var})"
                )
                output_vars.append(out_var)

            # Merge each Rule
            for rule in node.rules:
                r_var = f"rule_{rule_idx}"
                rule_idx += 1
                desc_esc = (rule.description or "").replace("'", "\\'")
                statements.append(
                    f"MERGE ({r_var}:DecisionRule {{id: '{rule.id}', description: '{desc_esc}'}})"
                )
                statements.append(f"MERGE ({d_var})-[:HAS_RULE]->({r_var})")

                # For each condition, link to the correct input by index
                # rule.conditions is a list of (clauseIndex, rawText)
                for (clause_index, cond_text) in rule.conditions:
                    c_var = f"cond_{cond_idx}"
                    cond_idx += 1
                    cond_val = (cond_text or "Unknown").replace("'", "\\'")
                    # We'll store the name=some sanitized version, or 'Clause_{index}'
                    # plus the raw condition expression as 'value'
                    c_name = f"clause_{clause_index}"
                    c_name_sanitized = self.sanitize(c_name)

                    statements.append(
                        f"MERGE ({c_var}:Condition {{name: '{c_name_sanitized}', value: '{cond_val}'}})"
                    )
                    statements.append(
                        f"MERGE ({r_var})-[:HAS_CONDITION]->({c_var})"
                    )

                    # If we have an input for that clause_index, link Condition->Input
                    if 0 <= clause_index < len(input_vars):
                        target_inp_var = input_vars[clause_index]
                        statements.append(
                            f"MERGE ({c_var})-[:CONDITION_ON]->({target_inp_var})"
                        )

                # Merge outcomes
                for out_text in rule.outcomes:
                    outv = f"outcome_{outcome_idx}"
                    outcome_idx += 1
                    statements.append(
                        f"MERGE ({outv}:DecisionOutcome {{name: '{out_text}'}})"
                    )
                    statements.append(
                        f"MERGE ({r_var})-[:HAS_OUTCOME]->({outv})"
                    )

        return "\n".join(statements)
        
# Example usage
# if __name__ == "__main__":
dmn_files = ['./nccn/Confirmed diagnosis and clinical stage.dmn']
bpmn_file = './nccn/NCCN Breast Cancer 2023.2.bpmn'

graph = DecisionGraph()
graph.parse_dmn(dmn_files)
graph.parse_bpmn(bpmn_file)

cypher_script = graph.generate_cypher()
print(cypher_script)


In [ ]:
import xml.etree.ElementTree as ET
import re
import json
from typing import List, Dict, Any

class OntologyBuilder:
    def __init__(self):
        self.decisions = {}

    def sanitize(self, text: str) -> str:
        """Sanitize text for node variable names by replacing invalid characters with underscores.
        If text is None, return 'Unknown'."""
        if text is None:
            return "Unknown"
        # Remove quotes if text is wrapped in them
        if text.startswith('\"') and text.endswith('\"'):
            text = text[2:-2]
        return re.sub(r'[^a-zA-Z0-9_]', '_', text)

    def is_unknown(self, text: str) -> bool:
        """Return True if text is None, empty, or equals 'Unknown' (case-insensitive)."""
        if text is None or text.strip() == "":
            return True
        return text.strip().lower() == "unknown"

    def parse_dmn(self, dmn_files: List[str]):
        NS = {'dmn': 'https://www.omg.org/spec/DMN/20191111/MODEL/'}

        for file_path in dmn_files:
            tree = ET.parse(file_path)
            root = tree.getroot()

            for decision_el in root.findall('dmn:decision', NS):
                decision_id = decision_el.get('id')
                decision_name = decision_el.get('name')

                if decision_id is None:
                    continue

                decision = {
                    'id': decision_id,
                    'name': decision_name,
                    'inputs': [],
                    'outputs': [],
                    'rules': [],
                    'dependsOn': []
                }

                # Collect dependencies (required decisions)
                for req_decision in decision_el.findall('.//dmn:requiredDecision', NS):
                    href = req_decision.get('href')
                    if href:
                        decision['dependsOn'].append(href.replace('#', ''))

                dtable = decision_el.find('dmn:decisionTable', NS)
                if dtable is None:
                    continue

                # Collect inputs in order
                inputs = dtable.findall('dmn:input', NS)
                for inp_el in inputs:
                    label = inp_el.get('label')
                    if label is not None:
                        decision['inputs'].append(label)

                # Collect outputs in order
                outputs = dtable.findall('dmn:output', NS)
                for out_el in outputs:
                    label = out_el.get('label') or out_el.get('name')
                    if label is not None:
                        decision['outputs'].append(label)

                # Collect rules from the decision table
                for rule_el in dtable.findall('dmn:rule', NS):
                    rule = {'conditions': {}, 'outcomes': {}}
                    input_entries = rule_el.findall('dmn:inputEntry', NS)
                    for idx, in_entry in enumerate(input_entries):
                        cond_text_el = in_entry.find('dmn:text', NS)
                        cond_text = cond_text_el.text if cond_text_el is not None else None
                        if cond_text is not None:
                            # Map the condition to the corresponding input label by index
                            if idx < len(decision['inputs']):
                                rule['conditions'][decision['inputs'][idx]] = cond_text
                            else:
                                rule['conditions'][f'input_{idx}'] = cond_text

                    output_entries = rule_el.findall('dmn:outputEntry', NS)
                    for idx, out_entry in enumerate(output_entries):
                        text_el = out_entry.find('dmn:text', NS)
                        outcome = text_el.text if text_el is not None else None
                        if outcome is not None and idx < len(decision['outputs']):
                            output_label = decision['outputs'][idx]
                            # Strip quotes for decision node reference if this is a go-to
                            if output_label == 'go-to':
                                outcome = outcome.strip('"')
                            rule['outcomes'][output_label] = outcome

                    decision['rules'].append(rule)

                self.decisions[decision_id] = decision
                print(decision)
                
    def generate_json(self) -> str:
        return json.dumps(self.decisions, indent=2)

    def generate_cypher(self) -> str:
        cypher = []
        # Global set to keep track of declared nodes
        declared_nodes = set()
        declared_params = set()
        
        # Process each Decision in the ontology
        for decision_id, decision in self.decisions.items():
            if decision_id is None:
                continue
                
            var_decision = self.sanitize(decision_id)
            if var_decision not in declared_nodes:
                cypher.append(f"MERGE ({var_decision}:Decision {{id:'{decision_id}', name:'{decision['name']}'}})")
                declared_nodes.add(var_decision)
            
            # Process inputs: ignore if unknown or None
            for input_name in decision['inputs']:
                if input_name is None or self.is_unknown(input_name):
                    continue
                var_input = self.sanitize(input_name)
                if var_input not in declared_params:
                    cypher.append(f"MERGE ({var_input}:ClinicalParameter {{name:'{input_name}'}})")
                    declared_params.add(var_input)
                cypher.append(f"MERGE ({var_decision})-[:HAS_INPUT]->({var_input})")
            
            # Link dependencies between decisions
            for dep in decision['dependsOn']:
                if dep is None:
                    continue
                var_dep = self.sanitize(dep)
                if var_dep not in declared_nodes:
                    cypher.append(f"MERGE ({var_dep}:Decision {{id:'{dep}'}})")
                    declared_nodes.add(var_dep)
                cypher.append(f"MERGE ({var_decision})-[:DEPENDS_ON]->({var_dep})")
            
            # Process rules
            for idx, rule in enumerate(decision['rules']):
                rule_node = f"rule_{var_decision}_{idx}"
                cypher.append(f"MERGE ({rule_node}:DecisionRule {{id:'{rule_node}'}})")
                cypher.append(f"MERGE ({var_decision})-[:HAS_RULE]->({rule_node})")
                
                # Process conditions in the rule
                for param, cond in rule['conditions'].items():
                    if param is None or cond is None or self.is_unknown(param):
                        continue
                    cond_sanitized = self.sanitize(cond)
                    param_sanitized = self.sanitize(param)
                    condition_node = f"cond_{rule_node}_{param_sanitized}"
                    cypher.append(f"MERGE ({condition_node}:Condition {{param:'{param_sanitized}', value:'{cond_sanitized}'}})")
                    cypher.append(f"MERGE ({rule_node})-[:HAS_CONDITION]->({condition_node})")
                    
                    # Only create a ClinicalParameter node if not already declared
                    if param_sanitized not in declared_params:
                        cypher.append(f"MERGE ({param_sanitized}:ClinicalParameter {{name:'{param}'}})")
                        declared_params.add(param_sanitized)
                    cypher.append(f"MERGE ({condition_node})-[:CONDITION_ON]->({param_sanitized})")
                
                # Process outcomes in the rule; ignore unknown or None outcomes
                for output_label, outcome in rule['outcomes'].items():
                    if outcome is None or self.is_unknown(outcome):
                        continue
                    outcome_sanitized = self.sanitize(outcome)
                    outcome_node = f"outcome_{rule_node}_{outcome_sanitized}"
                    
                    # Check if this is a transition outcome (go-to output)
                    is_transition = output_label == 'go-to'
                    outcome_type = "TransitionOutcome" if is_transition else "ActionRecommendation"
                    
                    cypher.append(f"MERGE ({outcome_node}:{outcome_type} {{name:'{outcome}'}})")
                    cypher.append(f"MERGE ({rule_node})-[:HAS_OUTCOME]->({outcome_node})")
                    
                    # For transition outcomes, create target decision node and relationship
                    if is_transition:
                        var_target = self.sanitize(outcome)
                        if var_target not in declared_nodes:
                            cypher.append(f"MERGE ({var_target}:Decision {{id:'{outcome}'}})")
                            declared_nodes.add(var_target)
                        cypher.append(f"MERGE ({outcome_node})-[:LEADS_TO_DECISION]->({var_target})")
        
        return "\n".join(cypher)

# Usage Example
# if __name__ == '__main__':
#     builder = OntologyBuilder()
#     builder.parse_dmn([
#         '/mnt/data/Confirmed diagnosis and clinical stage.dmn',
        # '/mnt/data/Evaluation for metastatic therapy.dmn',
        # '/mnt/data/First-line therapy for HR-HER2+.dmn',
        # '/mnt/data/First-line therapy for HR+HER2-.dmn',
        # '/mnt/data/First-line therapy for HR+HER2+.dmn'
#     ])

#     # Output JSON structured representation
#     ontology_json = builder.generate_json()
#     with open('ontology_representation.json', 'w') as f:
#         f.write(ontology_json)

#     # Output Cypher script
#     cypher_script = builder.generate_cypher()
#     with open('ontology_graph.cypher', 'w') as f:
#         f.write(cypher_script)

#     print("JSON and Cypher scripts have been generated.")

# # Usage Example
# if __name__ == '__main__':
builder = OntologyBuilder()
builder.parse_dmn([
    # './nccn/Confirmed diagnosis and clinical stage.dmn',
    './nccn/Evaluation for metastatic therapy.dmn',
    './nccn/First-line therapy for HR-HER2+.dmn',
    './nccn/First-line therapy for HR+HER2-.dmn',
    './nccn/First-line therapy for HR+HER2+.dmn'
])

# Output JSON structured representation
ontology_json = builder.generate_json()
with open('ontology_representation.json', 'w') as f:
    f.write(ontology_json)

# Output Cypher script
cypher_script = builder.generate_cypher()
with open('ontology_graph.cypher', 'w') as f:
    f.write(cypher_script)

print("JSON and Cypher scripts have been generated.")


In [ ]:
from probill.agents.mcp_host_agent import McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams
from autogen_agentchat.ui import Console
from autogen_agentchat.messages import TextMessage, MultiModalMessage

base_model = OpenAIChatCompletionClient(
    model="qwen2.5-coder:32b-instruct-q5_0",
    api_key="sk-proj-1234567890",
    base_url="http://10.0.40.49:11434/v1",
    model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": "unknown",
    "structured_output": True
    },
    temperature=0.0,
    top_p=1.0      
)

mcp_server_path = str("/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mcp_tools/mcp_neo4j_cypher/src/mcp_neo4j_cypher/")
cypher_sample = """```cypher
//Add a new patient node and their PatientData nodes
MERGE (p001:Patient {id: 'P001'})
ON CREATE SET p001.vocabulary_id = 'sacf', p001.concept_name = 'Patient P001', p001.domain_id = 'patient', p001.concept_class_id = 'Patient'
MERGE (p001)-[:HAS_DATA]->(pd001_1:PatientData {data_type: 'BoneDiseasePresent', value: 'false'})
ON CREATE SET pd001_1.vocabulary_id = 'sacf', pd001_1.concept_name = 'BoneDiseasePresent', pd001_1.domain_id = 'condition', pd001_1.concept_class_id = 'Condition'
MERGE (p001)-[:HAS_DATA]->(pd001_2:PatientData {data_type: 'ERStatus', value: 'positive'})
ON CREATE SET pd001_1.vocabulary_id = 'sacf', pd001_1.concept_name = 'ERStatus', pd001_1.domain_id = 'condition', pd001_1.concept_class_id = 'Condition'
//Update the patient node and their PatientData nodes
MATCH (p001:Patient {id: 'P001'})
SET p001.vocabulary_id = 'sacf', p001.concept_name = 'Patient P001', p001.domain_id = 'patient', p001.concept_class_id = 'Patient'
MATCH (p001)-[:HAS_DATA]->(pd001_1:PatientData {data_type: 'BoneDiseasePresent'})
SET pd001_1.vocabulary_id = 'sacf', pd001_1.concept_name = 'BoneDiseasePresent', pd001_1.domain_id = 'condition', pd001_1.concept_class_id = 'Condition', pd001_1.value = 'false'
//Delete a patient node and their PatientData nodes
MATCH (p:Patient {id: 'P001'})-[:HAS_DATA]->(pd:PatientData)
DETACH DELETE p, pd
```"""
              

system_prompt = f"You are Cypher script expert. Identify patient data from user's input. Follow the pattern of the sample Cypher script: \n{cypher_sample}\nGenerate a new Cypher script that merges the Patient node and their PatientData nodes. The new script should include the vocabulary_id, concept_name, domain_id, and concept_class_id properties for each node. The new script should also include the relationships between the Patient node and their PatientData nodes; One Patient Data node for one condition; When you remove a Patient entity, always remove its PatientData entities. Then, use the Cypher write tool to execute the script."

mcp_host_agent_1 = McpHostAgent(
    name="mcp_host_agent",
    model_client=base_model,
    description="A host agent that can use tools provided by MCP servers.",
    server_params=StdioServerParams(
        command="uv", 
        args=[
            "--directory",
            mcp_server_path,
            "run",
            "server.py",
            "--db-url",
            "bolt://10.0.40.49:7687",
            "--username", 
            "neo4j",
            "--password",
            "sacf_sacf",
            "--database",
            "neo4j",
        ]
    ),
    model_client_stream=True,
    system_message=system_prompt,
    reflect_on_tool_use=True,
)

user_message = TextMessage(
    source="user",
    content = "Create a new patient node and their PatientData nodes. The patient id is P010. The BoneDiseasePresent is false, ERStatus is positive, PRStatus is positive, HER2Status is negative."
    )
system_message = TextMessage(
    source="system",
    content = system_prompt
    )

task = [system_message, user_message]
print(task)

In [ ]:
# task = "Patient: P009 PRStats positive, HER2Status negative, BoneDiseasePresent false, ERStatus positive"
result = await Console(mcp_host_agent_1.run_stream(task="Remove patient data: HER2Status, PRStatus, ERStatus from patient P010"), output_stats=True)

In [ ]:
print(mcp_host_agent_1.dump_component().model_dump_json(indent=2))

In [ ]:
from probill.agents.mcp_host_agent import McpHostAgent
from autogen_ext.models.ollama import OllamaChatCompletionClient
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import SseServerParams
from autogen_agentchat.ui import Console
from autogen_agentchat.agents import AssistantAgent

base_model = OpenAIChatCompletionClient(
    model="qwen2.5-coder:32b-instruct-q5_0",
    api_key="sk-proj-1234567890",
    base_url="http://10.0.40.49:11434/v1",
    model_info = {
    "vision": False,
    "function_calling": True,
    "json_output": True,
    "family": "unknown",
    "structured_output": True
    },
    temperature=0.0        
)

mcp_host_agent = McpHostAgent(
    name="mcp_host_agent",
    model_client=base_model,
    description="A host agent that can use tools provided by MCP servers.",
    server_params=SseServerParams(
        url="http://localhost:9003/sse",
        timeout=60,
    )
)

In [ ]:
result = await Console(mcp_host_agent.run_stream(task="Hi"))

In [ ]:
print(mcp_host_agent.dump_component().model_dump_json(indent=2))

In [ ]:
result = await Console(mcp_host_agent.run_stream(task="What do you have in the knowledge graph?"),output_stats=True)

In [ ]:
for msg in result.messages:
    print(msg)

In [ ]:
agent_json = mcp_host_agent.dump_component().model_dump_json(indent=4)
print(agent_json)

In [ ]:
{
    "provider": "probill.agents.mcp_host_agent.McpHostAgent",
    "component_type": "agent",
    "version": 1,
    "component_version": 1,
    "description": "McpHostAgent is an agent that can connect to MCP servers and manage tools, resources, and prompts.\n    It acts as a host that can understand and utilize tools provided by MCP servers through either\n    SseMcpToolAdapter or StdioMcpToolAdapter.",
    "label": "McpHostAgent",
    "config": {
        "name": "mcp_host_agent",
        "model_client": {
            "provider": "autogen_ext.models.ollama.OllamaChatCompletionClient",
            "component_type": "model",
            "version": 1,
            "component_version": 1,
            "description": "Chat completion client for Ollama hosted models.",
            "label": "OllamaChatCompletionClient",
            "config": {
                "model": "qwen2.5-coder:32b-instruct-q5_0",
                "host": "http://10.0.40.49:11434",
                "follow_redirects": true,
                "model_info": {
                    "vision": false,
                    "function_calling": true,
                    "json_output": true,
                    "family": "unknown"
                }
            }
        },
        "description": "McpHostAgent is an agent that can connect to MCP servers and manage tools, resources, and prompts.\n    It acts as a host that can understand and utilize tools provided by MCP servers through either\n    SseMcpToolAdapter or StdioMcpToolAdapter.",
        "server_params": {
            "command": "uv",
            "args": [
                "--directory",
                "/workspaces/OrchestrAI-autogen/python/packages/probill/src/probill/mcp_tools/mcp-neo4j/servers/mcp-neo4j-cypher",
                "run",
                "mcp-neo4j-cypher",
                "--db-url",
                "bolt://10.0.40.49:7687",
                "--username",
                "neo4j",
                "--password",
                "sacf_sacf"
            ],
            "encoding": "utf-8",
            "encoding_error_handler": "strict"
        }
    }
}

In [ ]:
from enum import Enum
from typing import List, Dict, Optional
from datetime import datetime

### Enums for Fixed Values
class PerformanceStatus(Enum):
    ECOG0 = 0
    ECOG1 = 1
    ECOG2 = 2
    ECOG3 = 3
    ECOG4 = 4

class TumorType(Enum):
    PHYLLODES = "Phyllodes"
    INVASIVE = "Invasive"
    DCIS = "DCIS"

class TherapyType(Enum):
    SURGERY = "Surgery"
    CHEMOTHERAPY = "Chemotherapy"
    ENDOCRINE = "Endocrine"
    TARGETED = "Targeted"
    BONE_MODIFYING = "BoneModifying"
    SUPPORTIVE = "SupportiveCare"

### Core Classes
class Patient:
    """Represents a patient with attributes relevant to decision-making."""
    def __init__(self, patient_id: str, age: int, performance_status: PerformanceStatus, preferences: str):
        self.patient_id = patient_id  # Unique identifier for Neo4j
        self.age = age
        self.performance_status = performance_status
        self.preferences = preferences
        self.clinical_presentations: List[ClinicalPresentation] = []

    def add_clinical_presentation(self, presentation: 'ClinicalPresentation') -> None:
        self.clinical_presentations.append(presentation)

class ClinicalPresentation:
    """Captures the clinical state influencing decisions."""
    def __init__(self, presentation_id: str, tumor_type: TumorType, stage: str, 
                 biomarkers: Dict[str, str], prior_treatment: List[str], 
                 disease_sites: List[str], visceral_crisis: bool):
        self.presentation_id = presentation_id  # Unique ID for Neo4j
        self.tumor_type = tumor_type
        self.stage = stage
        self.biomarkers = biomarkers
        self.prior_treatment = prior_treatment
        self.disease_sites = disease_sites
        self.visceral_crisis = visceral_crisis
        self.diagnostic_procedures: List[DiagnosticProcedure] = []

    def add_diagnostic_procedure(self, procedure: 'DiagnosticProcedure') -> None:
        self.diagnostic_procedures.append(procedure)

class DiagnosticProcedure:
    """Provides data inputs for the decision process."""
    def __init__(self, procedure_id: str, type: str, result: str, date_performed: str):
        self.procedure_id = procedure_id  # Unique ID for Neo4j
        self.type = type
        self.result = result
        self.date_performed = date_performed  # e.g., "2023-01-01"

class TreatmentPlan:
    """Defines a plan influenced by the decision process."""
    def __init__(self, plan_id: str, goal: str, start_date: str):
        self.plan_id = plan_id  # Unique ID for Neo4j
        self.goal = goal
        self.start_date = start_date
        self.interventions: List[Intervention] = []

    def add_intervention(self, intervention: 'Intervention') -> None:
        self.interventions.append(intervention)

class Intervention:
    """A specific action recommended by the decision process."""
    def __init__(self, intervention_id: str, type: TherapyType, regimen: str, 
                 risk_benefit_assessment: Dict[str, str]):
        self.intervention_id = intervention_id  # Unique ID for Neo4j
        self.type = type
        self.regimen = regimen
        self.risk_benefit_assessment = risk_benefit_assessment

class DecisionStep:
    """A single step in the decision process, evaluating a specific criterion."""
    def __init__(self, step_id: str, description: str, criterion: 'Criterion', 
                 input_data: Dict[str, str], evaluation_result: str):
        self.step_id = step_id  # Unique ID for Neo4j
        self.description = description  # e.g., "Check survival threshold"
        self.criterion = criterion
        self.input_data = input_data  # e.g., {"ExpectedSurvival": "6 months"}
        self.evaluation_result = evaluation_result  # e.g., "Met"

class Criterion:
    """A condition evaluated in a decision step."""
    def __init__(self, criterion_id: str, name: str, threshold: Optional[str] = None):
        self.criterion_id = criterion_id  # Unique ID for Neo4j
        self.name = name  # e.g., "SurvivalThreshold"
        self.threshold = threshold  # e.g., "3 months"

class DecisionPoint:
    """Models the decision process as a series of steps."""
    def __init__(self, decision_id: str, time_point: str):
        self.decision_id = decision_id  # Unique ID for Neo4j
        self.time_point = time_point  # e.g., "2023-01-01"
        self.steps: List[DecisionStep] = []
        self.recommended_interventions: List[Intervention] = []

    def add_step(self, step: DecisionStep) -> None:
        self.steps.append(step)

    def add_recommended_intervention(self, intervention: Intervention) -> None:
        self.recommended_interventions.append(intervention)

    def evaluate(self, patient: Patient, presentation: ClinicalPresentation) -> None:
        """Simulate the decision process, storing steps rather than final outcome."""
        # Step 1: Check visceral crisis
        visceral_step = DecisionStep(
            step_id=f"{self.decision_id}_step1",
            description="Evaluate visceral crisis",
            criterion=Criterion(criterion_id="C1", name="VisceralCrisis", threshold="True"),
            input_data={"VisceralCrisis": str(presentation.visceral_crisis)},
            evaluation_result="Met" if presentation.visceral_crisis else "Not Met"
        )
        self.add_step(visceral_step)

        # Step 2: Assess performance status
        perf_step = DecisionStep(
            step_id=f"{self.decision_id}_step2",
            description="Assess performance status",
            criterion=Criterion(criterion_id="C2", name="PerformanceStatusThreshold", 
                              threshold="ECOG2"),
            input_data={"PerformanceStatus": patient.performance_status.name},
            evaluation_result="Met" if patient.performance_status.value <= 2 else "Not Met"
        )
        self.add_step(perf_step)

        # Step 3: Check bone metastases (example from BINV-20)
        bone_step = DecisionStep(
            step_id=f"{self.decision_id}_step3",
            description="Check for bone metastases",
            criterion=Criterion(criterion_id="C3", name="BoneMetastases", threshold="Present"),
            input_data={"DiseaseSites": ", ".join(presentation.disease_sites)},
            evaluation_result="Met" if "bone" in presentation.disease_sites else "Not Met"
        )
        self.add_step(bone_step)

        # Recommend interventions based on steps (example logic)
        if visceral_step.evaluation_result == "Met":
            intervention = Intervention(
                intervention_id=f"{self.decision_id}_int1",
                type=TherapyType.CHEMOTHERAPY,
                regimen="Paclitaxel",
                risk_benefit_assessment={"Benefit": "High", "Risk": "Moderate"}
            )
            self.add_recommended_intervention(intervention)
        elif bone_step.evaluation_result == "Met" and perf_step.evaluation_result == "Met":
            intervention = Intervention(
                intervention_id=f"{self.decision_id}_int2",
                type=TherapyType.BONE_MODIFYING,
                regimen="Denosumab",
                risk_benefit_assessment={"Benefit": "Moderate", "Risk": "Low"}
            )
            self.add_recommended_intervention(intervention)

### Example Usage
if __name__ == "__main__":
    # Create a patient
    patient = Patient(
        patient_id="P001", age=60, performance_status=PerformanceStatus.ECOG1, 
        preferences="Prefer active treatment"
    )

    # Define a clinical presentation
    presentation = ClinicalPresentation(
        presentation_id="CP001", tumor_type=TumorType.INVASIVE, stage="IV",
        biomarkers={"HR": "positive", "HER2": "negative"}, prior_treatment=["Endocrine"],
        disease_sites=["bone"], visceral_crisis=False
    )
    patient.add_clinical_presentation(presentation)

    # Add a diagnostic procedure
    bone_scan = DiagnosticProcedure(
        procedure_id="DP001", type="Bone Scan", result="Metastases present", 
        date_performed="2023-01-01"
    )
    presentation.add_diagnostic_procedure(bone_scan)

    # Create a decision point
    decision = DecisionPoint(decision_id="D001", time_point="2023-01-02")
    decision.evaluate(patient, presentation)

    # Print the decision process
    print(f"Decision Process for {decision.decision_id} at {decision.time_point}:")
    for step in decision.steps:
        print(f"Step {step.step_id}: {step.description}")
        print(f"  Criterion: {step.criterion.name} (Threshold: {step.criterion.threshold})")
        print(f"  Input: {step.input_data}")
        print(f"  Result: {step.evaluation_result}")
    print("\nRecommended Interventions:")
    for intervention in decision.recommended_interventions:
        print(f"  {intervention.intervention_id}: {intervention.type.value} - {intervention.regimen}")

In [ ]:
from neo4j import GraphDatabase
from typing import List, Dict, Optional

class DecisionPoint:
    """Represents a decision node in the reasoning process."""
    def __init__(self, id: str, description: str, condition_query: str, 
                 if_met: Optional['TherapyOption' or 'DecisionPoint'] = None, 
                 if_not_met: Optional['TherapyOption' or 'DecisionPoint'] = None):
        self.id = id
        self.description = description
        self.condition_query = condition_query  # Cypher query to evaluate the condition
        self.if_met = if_met
        self.if_not_met = if_not_met

    def evaluate(self, patient_data: List[Dict[str, str]], driver) -> tuple[Optional['TherapyOption' or 'DecisionPoint'], List[str]]:
        """Evaluates the decision point by querying the graph and returns the next step and any missing data."""
        missing_data = []
        print(f"Evaluating {self.id} with query: {self.condition_query}")  # Debug print
        with driver.session() as session:
            # Ensure patient_data is passed as a parameter
            result = session.run(self.condition_query, patient_data=[{"data_type": d["data_type"], "value": d["value"]} for d in patient_data])
            record = result.single()
            if not record or (record["result"] is None and not self._has_matching_data(result)):
                missing_data = self._infer_missing_data(patient_data, self.condition_query)
                print(f"No result or no matching data for {self.id}, missing data: {missing_data}")  # Debug print
                return None, missing_data

            condition_met = record["result"]
            print(f"Condition met for {self.id}: {condition_met}")  # Debug print
            if condition_met:
                return self.if_met, []
            return self.if_not_met, []

    def _has_matching_data(self, result):
        """Check if the query result contains any matching data."""
        return result.peek() is not None  # True if there is at least one row

    def _infer_missing_data(self, patient_data: List[Dict[str, str]], condition_query: str) -> List[str]:
        """Infer missing data based on the condition query."""
        available_data = {d["data_type"] for d in patient_data}
        import re
        match = re.search(r'data\.data_type = "([^"]+)"', condition_query)
        required_field = match.group(1) if match else None
        if required_field and required_field not in available_data:
            return [required_field]
        return []

    def __str__(self):
        return f"DecisionPoint({self.id}: {self.description})"

class TherapyOption:
    """Represents a recommended treatment."""
    def __init__(self, id: str, type: str, regimen: str, outcome: str = "", next_step: str = ""):
        self.id = id
        self.type = type
        self.regimen = regimen
        self.outcome = outcome
        self.next_step = next_step

    def __str__(self):
        return f"TherapyOption({self.id}: {self.type} - {self.regimen}, Outcome: {self.outcome}, Next: {self.next_step})"

class DecisionLoader:
    """Loads and evaluates the decision tree from Neo4j, handling missing data."""
    def __init__(self, uri: str, user: str, password: str):
        """Initialize the Neo4j driver."""
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        """Close the Neo4j driver connection."""
        self.driver.close()

    def persist_decision_tree(self):
        """Persist the BINV-21 decision tree into Neo4j, including the Page node."""
        with self.driver.session() as session:
            # Clear existing data for a fresh start
            session.run("MATCH (n) DETACH DELETE n")

            # Create Page node for BINV-21
            session.run("""
            CREATE (p1:Page {id: 'BINV-21', title: 'SYSTEMIC TREATMENT OF RECURRENT UNRESECTABLE (LOCAL OR REGIONAL) OR STAGE IV (M1) DISEASE: ER- AND/OR PR-POSITIVE; HER2-NEGATIVEd'})
            """)

            # Decision Point 1: Visceral Crisis
            session.run("""
            MATCH (p1:Page {id: 'BINV-21'})
            CREATE (dp1:DecisionPoint {id: 'DP1', description: 'Does the patient have a visceral crisis?', 
                                     condition_query: 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "VisceralCrisis" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN HEAD(COLLECT(data.value)) = "true" THEN true ELSE false END AS result'})
            CREATE (t1:TherapyOption {id: 'T1', type: 'Systemic Therapy', 
                                     regimen: 'Consider initial systemic therapy (BINV-Q)', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (dp2:DecisionPoint {id: 'DP2', description: 'Assess prior endocrine therapy and menopausal status', 
                                     condition_query: 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "PriorEndocrineTherapyWithin1Year" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN HEAD(COLLECT(CASE WHEN data.value = "true" THEN true ELSE false END)) THEN true ELSE false END AS result'})
            CREATE (p1)-[:HAS_DECISION_POINT]->(dp1)
            CREATE (dp1)-[:EVALUATES]->(dp1)
            CREATE (dp1)-[:IF_MET]->(t1)
            CREATE (dp1)-[:IF_NOT_MET]->(dp2)
            """)

            # Decision Point 2: Prior Endocrine Therapy Within 1 Year
            session.run("""
            MATCH (dp2:DecisionPoint {id: 'DP2'})
            CREATE (dp3:DecisionPoint {id: 'DP3', description: 'Menopausal status for prior therapy within 1 year', 
                                     condition_query: 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "PriorEndocrineTherapyWithin1Year" AND data.value = "true" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN true THEN true ELSE false END AS result'})
            CREATE (dp4:DecisionPoint {id: 'DP4', description: 'Menopausal status for no prior therapy within 1 year', 
                                     condition_query: 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "MenopausalStatus" AND data.value = "postmenopausal" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN true THEN true ELSE false END AS result'})
            CREATE (dp2)-[:EVALUATES]->(dp2)
            CREATE (dp2)-[:IF_MET]->(dp3)
            CREATE (dp2)-[:IF_NOT_MET]->(dp4)
            """)

            # Decision Point 3: Menopausal Status (Prior Therapy Within 1 Year)
            session.run("""
            MATCH (dp3:DecisionPoint {id: 'DP3'})
            CREATE (t2:TherapyOption {id: 'T2', type: 'Ovarian Ablation/Suppression + Systemic Therapy', 
                                     regimen: 'See BINV-P', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (t3:TherapyOption {id: 'T3', type: 'Systemic Therapy', 
                                     regimen: 'See BINV-P', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (dp3)-[:EVALUATES]->(dp3)
            CREATE (dp3)-[:IF_MET]->(t2)
            CREATE (dp3)-[:IF_NOT_MET]->(t3)
            WITH dp3
            SET dp3.condition_query = 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "MenopausalStatus" AND data.value = "premenopausal" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN true THEN true ELSE false END AS result'
            """)

            # Decision Point 4: Menopausal Status (No Prior Therapy Within 1 Year)
            session.run("""
            MATCH (dp4:DecisionPoint {id: 'DP4'})
            CREATE (t4:TherapyOption {id: 'T4', type: 'Ovarian Ablation/Suppression + Systemic Therapy', 
                                     regimen: 'See BINV-P', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (t5:TherapyOption {id: 'T5', type: 'Selective ER Modulators', 
                                     regimen: 'See BINV-P', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (t6:TherapyOption {id: 'T6', type: 'Systemic Therapy', 
                                     regimen: 'See BINV-P', 
                                     outcome: 'Continue until progression or unacceptable toxicity', 
                                     next_step: 'If progression, see BINV-22'})
            CREATE (dp4)-[:EVALUATES]->(dp4)
            CREATE (dp4)-[:IF_MET]->(t6)  // Postmenopausal -> Systemic Therapy
            CREATE (dp4)-[:IF_NOT_MET]->(t4)  // Premenopausal -> Ovarian Ablation/Suppression
            CREATE (dp4)-[:ALTERNATIVE]->(t5)
            WITH dp4
            SET dp4.condition_query = 'UNWIND $patient_data AS data WITH data WHERE data.data_type = "MenopausalStatus" AND data.value = "postmenopausal" RETURN CASE WHEN COUNT(data) = 0 THEN null WHEN true THEN true ELSE false END AS result'
            """)

            # Additional Notes and Relationships to Page
            session.run("""
            CREATE (n1:Note {id: 'N1', content: 'See Principles of Biomarker Testing (BINV-A)'})
            CREATE (n2:Note {id: 'N2', content: 'If progression on initial endocrine therapy, switch to different endocrine therapy option (See BINV-T)'})
            CREATE (n3:Note {id: 'N3', content: 'Switch to endocrine-based therapy acceptable after stabilization/response (See BINV-P)'})
            CREATE (n4:Note {id: 'N4', content: 'All recommendations category 2A unless indicated; clinical trials encouraged'})
            WITH n1, n2, n3, n4
            MATCH (p1:Page {id: 'BINV-21'})
            CREATE (p1)-[:HAS_NOTE]->(n1)
            CREATE (p1)-[:HAS_NOTE]->(n2)
            CREATE (p1)-[:HAS_NOTE]->(n3)
            CREATE (p1)-[:HAS_NOTE]->(n4)
            """)

    def load_decision_tree(self, decision_id: str) -> Optional[DecisionPoint]:
        """Loads a decision point and its dependencies from Neo4j."""
        query = """
        MATCH (dp:DecisionPoint {id: $decision_id})
        OPTIONAL MATCH (dp)-[:IF_MET]->(met)
        OPTIONAL MATCH (dp)-[:IF_NOT_MET]->(not_met)
        OPTIONAL MATCH (dp)-[:ALTERNATIVE]->(alt)
        RETURN dp, met, not_met, alt
        """
        with self.driver.session() as session:
            result = session.run(query, decision_id=decision_id)
            record = result.single()
            if not record:
                return None

            dp_node = record["dp"]
            met_node = record["met"]
            not_met_node = record["not_met"]
            alt_node = record["alt"]

            # Create DecisionPoint with dynamic condition query
            if_met = self._create_node_instance(met_node)
            if_not_met = self._create_node_instance(not_met_node)
            alternative = self._create_node_instance(alt_node) if alt_node else None

            dp = DecisionPoint(dp_node["id"], dp_node["description"], dp_node["condition_query"], if_met, if_not_met)
            if alternative and isinstance(dp.if_not_met, TherapyOption):
                dp.if_not_met = [dp.if_not_met, alternative]  # List for multiple options
            elif alternative:
                dp.if_not_met = alternative  # Override if no prior not_met
            return dp

    def _create_node_instance(self, node) -> Optional[DecisionPoint or TherapyOption]:
        """Creates a Python instance from a Neo4j node."""
        if not node:
            return None
        if "TherapyOption" in node.labels:
            return TherapyOption(node["id"], node["type"], node["regimen"], node["outcome"], node["next_step"])
        if "DecisionPoint" in node.labels:
            return self.load_decision_tree(node["id"])
        return None

    def evaluate_patient(self, patient_id: str, start_decision_id: str) -> str:
        """Evaluates the decision tree for a patient, reporting missing data."""
        # Fetch patient data
        patient_query = """
        MATCH (pd:PatientData {patient_id: $patient_id})
        RETURN pd.data_type AS data_type, pd.value AS value
        """
        with self.driver.session() as session:
            patient_result = session.run(patient_query, patient_id=patient_id)
            patient_data = [{"data_type": record["data_type"], "value": record["value"]} 
                          for record in patient_result]

        # Load and evaluate decision tree
        current = self.load_decision_tree(start_decision_id)
        if not current:
            return "Decision tree not found"

        steps = []
        missing_data = []
        while isinstance(current, DecisionPoint):
            steps.append(str(current))
            next_step, new_missing = current.evaluate(patient_data, self.driver)
            missing_data.extend(new_missing)
            if missing_data:
                return "\n".join(steps) + f"\nMissing required data: {', '.join(set(missing_data))}"
            if isinstance(next_step, TherapyOption):
                steps.append(str(next_step))
                return "\n".join(steps) + f"\nRecommended Therapy: {next_step.type} - {next_step.regimen}"
            elif next_step is None:
                return "\n".join(steps) + "\nNo recommendation determined"
            current = next_step
        return "\n".join(steps) + "\nNo recommendation determined"

# Example Usage with Different Inputs
if __name__ == "__main__":
    # Initialize loader
    loader = DecisionLoader("bolt://10.0.40.49:7687", "neo4j", "sacf_sacf")

    # Persist the decision tree with the Page node
    loader.persist_decision_tree()

    # Test cases with different patient data
    test_cases = [
        {
            "patient_id": "P001",
            "data": [("VisceralCrisis", "false"), ("PriorEndocrineTherapyWithin1Year", "false")]
        },
        {
            "patient_id": "P002",
            "data": [("VisceralCrisis", "true")]
        },
        {
            "patient_id": "P003",
            "data": [("VisceralCrisis", "false"), ("PriorEndocrineTherapyWithin1Year", "true"), ("MenopausalStatus", "premenopausal")]
        },
        {
            "patient_id": "P004",
            "data": [("VisceralCrisis", "false"), ("PriorEndocrineTherapyWithin1Year", "false"), ("MenopausalStatus", "postmenopausal")]  # Missing MenopausalStatus
        }
    ]

    for test in test_cases:
        patient_id = test["patient_id"]
        with loader.driver.session() as session:
            # Clear previous data for this patient
            session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
            for data_type, value in test["data"]:
                session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: $data_type, value: $value})",
                           patient_id=patient_id, data_type=data_type, value=value)

        # Evaluate patient
        recommendation = loader.evaluate_patient(patient_id, "DP1")
        print(f"\nEvaluation for Patient {patient_id}:")
        print(recommendation)
        print("-"*100,"\n\n")
    # Close connection
    loader.close()

In [ ]:
from neo4j import GraphDatabase
from typing import List, Dict, Optional, Tuple

class DecisionPoint:
    def __init__(self, id: str, description: str, condition_query: str, 
                 if_met: Optional['TherapyOption' or 'DecisionPoint' or 'Page'] = None, 
                 if_not_met: Optional['TherapyOption' or 'DecisionPoint' or 'Page'] = None):
        self.id = id
        self.description = description
        self.condition_query = condition_query
        self.if_met = if_met
        self.if_not_met = if_not_met

    def evaluate(self, patient_data: List[Dict[str, str]], session) -> tuple[Optional['TherapyOption' or 'DecisionPoint' or 'Page'], List[str]]:
        missing_data = []
        result = session.run(self.condition_query, patient_data=[{"data_type": d["data_type"], "value": d["value"]} for d in patient_data])
        record = result.single()
        if not record or record["result"] is None:
            missing_data = self._infer_missing_data(patient_data, self.condition_query)
            return None, missing_data
        condition_met = record["result"]
        if condition_met:
            return self.if_met, []
        return self.if_not_met, []

    def _infer_missing_data(self, patient_data: List[Dict[str, str]], condition_query: str) -> List[str]:
        available_data = {d["data_type"] for d in patient_data}
        import re
        match = re.search(r'data\.data_type = "([^"]+)"', condition_query)
        required_field = match.group(1) if match else None
        if required_field and required_field not in available_data:
            return [required_field]
        return []

    def __str__(self):
        return f"DecisionPoint({self.id}: {self.description})"

class TherapyOption:
    def __init__(self, id: str, type: str, regimen: str, outcome: str = "", next_step: str = ""):
        self.id = id
        self.type = type
        self.regimen = regimen
        self.outcome = outcome
        self.next_step = next_step

    def __str__(self):
        return f"TherapyOption({self.id}: {self.type} - {self.regimen}, Outcome: {self.outcome}, Next: {self.next_step})"

    def __eq__(self, other):
        if isinstance(other, TherapyOption):
            return self.id == other.id
        return False

class Page:
    def __init__(self, id: str, title: str):
        self.id = id
        self.title = title

    def __str__(self):
        return f"Page({self.id}: {self.title})"

class DecisionLoader:
    def __init__(self, uri: str, user: str, password: str):
        self.driver = GraphDatabase.driver(uri, auth=(user, password))

    def close(self):
        self.driver.close()

    def load_decision_tree(self, decision_id: str, session) -> Optional[DecisionPoint]:
        query = """
        MATCH (dp:DecisionPoint {id: $decision_id})
        OPTIONAL MATCH (dp)-[:IF_MET]->(met)
        OPTIONAL MATCH (dp)-[:IF_NOT_MET]->(not_met)
        RETURN dp, met, not_met
        LIMIT 1
        """
        result = session.run(query, decision_id=decision_id)
        record = result.single()
        if not record:
            return None
        dp_node = record["dp"]
        met_node = record["met"]
        not_met_node = record["not_met"]
        if_met = self._create_node_instance(met_node, session)
        if_not_met = self._create_node_instance(not_met_node, session)
        return DecisionPoint(dp_node["id"], dp_node["description"], dp_node["condition_query"], if_met, if_not_met)

    def _create_node_instance(self, node, session) -> Optional[DecisionPoint or TherapyOption or Page]:
        if not node:
            return None
        if "TherapyOption" in node.labels:
            return TherapyOption(node["id"], node["type"], node["regimen"], node["outcome"], node["next_step"])
        if "DecisionPoint" in node.labels:
            return self.load_decision_tree(node["id"], session)
        if "Page" in node.labels:
            return Page(node["id"], node["title"])
        return None

    def process_next_step(self, current: DecisionPoint, patient_data: List[Dict[str, str]], session, steps: List[str], 
                         therapy_options: List[TherapyOption], missing_data: List[str], iteration: int) -> Tuple[Optional[DecisionPoint], List[str], List[TherapyOption], List[str], int]:
        print(f"current step:  {current}")
        steps.append(str(current))
        next_step, new_missing = current.evaluate(patient_data, session)
        missing_data.extend(new_missing)
        if missing_data:
            return None, steps, therapy_options, missing_data, iteration

        if isinstance(next_step, TherapyOption):
            steps.append(str(next_step))
            if next_step not in therapy_options:  # Avoid duplicates
                therapy_options.append(next_step)
            # Check for next decision point (e.g., DP5 for progression)
            query = """
            MATCH (t:TherapyOption {id: $therapy_id})-[:HAS_DECISION_POINT]->(dp:DecisionPoint)
            RETURN dp
            LIMIT 1
            """
            result = session.run(query, therapy_id=next_step.id)
            record = result.single()
            if record:
                return self.load_decision_tree(record["dp"]["id"], session), steps, therapy_options, missing_data, iteration
            return None, steps, therapy_options, missing_data, iteration

        elif isinstance(next_step, Page):
            steps.append(f"Transition to {next_step}")
            # Query the first decision point under the page to start the sequence
            page_id = next_step.id
            query = """
            MATCH (p:Page {id: $page_id})-[:HAS_DECISION_POINT]->(dp:DecisionPoint)
            RETURN dp
            LIMIT 1
            """
            result = session.run(query, page_id=page_id)
            record = result.single()
            if record:
                return self.load_decision_tree(record["dp"]["id"], session), steps, therapy_options, missing_data, iteration
            return None, steps, therapy_options, missing_data, iteration

        elif next_step is None:
            return None, steps, therapy_options, missing_data, iteration

        return next_step, steps, therapy_options, missing_data, iteration

    def evaluate_patient(self, patient_id: str, start_page_id: str) -> dict:
        with self.driver.session() as session:
            # Fetch patient data
            patient_query = """
            MATCH (pd:PatientData {patient_id: $patient_id})
            RETURN pd.data_type AS data_type, pd.value AS value
            """
            patient_result = session.run(patient_query, patient_id=patient_id)
            patient_data = [{"data_type": record["data_type"], "value": record["value"]} for record in patient_result]

            # Start with the first decision point under the specified page
            query = """
            MATCH (p:Page {id: $page_id})-[:HAS_DECISION_POINT]->(dp:DecisionPoint)
            RETURN dp
            LIMIT 1
            """
            result = session.run(query, page_id=start_page_id)
            record = result.single()
            if not record:
                return {"error": f"No decision points found for page {start_page_id}"}
            current = self.load_decision_tree(record["dp"]["id"], session)

            if not current:
                return {"error": "Decision tree not found"}
            
            steps = []
            therapy_options = []  # Collect all matching therapy options
            missing_data = []
            max_iterations = 10  # Prevent infinite loop
            iteration = 0
            while isinstance(current, DecisionPoint) and iteration < max_iterations:
                current, steps, therapy_options, missing_data, iteration = self.process_next_step(
                    current, patient_data, session, steps, therapy_options, missing_data, iteration
                )
                iteration += 1
            result = {"steps": steps}
            if missing_data:
                result["missing_data"] = missing_data
            if therapy_options:
                therapies_str = "\n".join([str(t) for t in therapy_options])
                result["therapies"] = therapies_str
            return result

if __name__ == "__main__":
    # Initialize loader
    loader = DecisionLoader("bolt://10.0.40.49:7687", "neo4j", "sacf_sacf")

    # Test Case 1: BINV-21 with No Progression or Toxicity
    patient_id = "P001"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'VisceralCrisis', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PriorEndocrineTherapyWithin1Year', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'MenopausalStatus', value: 'postmenopausal'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ProgressionOrToxicity', value: 'false'})", patient_id=patient_id)

    print("Test Case 1: BINV-21 with No Progression or Toxicity")
    recommendation = loader.evaluate_patient(patient_id, "BINV-21")
    import json
    print(json.dumps(recommendation, indent=4))
    print("\n" + "-"*50 + "\n")

    # Test Case 2: BINV-21 with Progression or Toxicity (No Visceral Crisis)
    patient_id = "P002"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'VisceralCrisis', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PriorEndocrineTherapyWithin1Year', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'MenopausalStatus', value: 'postmenopausal'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ProgressionOrToxicity', value: 'true'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'EndocrineRefractory', value: 'false'})", patient_id=patient_id)

    print("Test Case 2: BINV-21 with Progression or Toxicity (No Visceral Crisis)")
    recommendation = loader.evaluate_patient(patient_id, "BINV-21")
    print(json.dumps(recommendation, indent=4))
    print("\n" + "-"*50 + "\n")

    # Test Case 3: BINV-22 with Visceral Crisis
    patient_id = "P003"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'VisceralCrisis', value: 'true'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ProgressionOrToxicity', value: 'true'})", patient_id=patient_id)

    print("Test Case 3: BINV-22 with Visceral Crisis")
    recommendation = loader.evaluate_patient(patient_id, "BINV-21")
    print(json.dumps(recommendation, indent=4))

# Test Case 4: BINV-20 with No Bone Disease, ER/PR-positive, HER2-negative, No Progression
    patient_id = "P004"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'BoneDiseasePresent', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ERStatus', value: 'positive'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PRStatus', value: 'positive'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'HER2Status', value: 'negative'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'VisceralCrisis', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PriorEndocrineTherapyWithin1Year', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'MenopausalStatus', value: 'postmenopausal'})", patient_id=patient_id)
        # session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ProgressionOrToxicity', value: 'false'})", patient_id=patient_id)

    print("Test Case 4: BINV-20 with No Bone Disease, ER/PR-positive, HER2-negative, No Progression")
    recommendation = loader.evaluate_patient(patient_id, "BINV-20")
    import json
    print(json.dumps(recommendation, indent=4))
    print("\n" + "-"*50 + "\n")

    # Test Case 2: BINV-20 with Bone Disease, ER/PR-positive, HER2-negative, Progression
    patient_id = "P005"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'BoneDiseasePresent', value: 'true'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ERStatus', value: 'positive'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PRStatus', value: 'positive'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'HER2Status', value: 'negative'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'VisceralCrisis', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PriorEndocrineTherapyWithin1Year', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'MenopausalStatus', value: 'postmenopausal'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ProgressionOrToxicity', value: 'true'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'EndocrineRefractory', value: 'false'})", patient_id=patient_id)

    print("Test Case 5: BINV-20 with Bone Disease, ER/PR-positive, HER2-negative, Progression")
    recommendation = loader.evaluate_patient(patient_id, "BINV-20")
    print(json.dumps(recommendation, indent=4))
    print("\n" + "-"*50 + "\n")

    # Test Case 3: BINV-20 with No Bone Disease, ER/PR-negative, HER2-positive
    patient_id = "P006"
    with loader.driver.session() as session:
        session.run("MATCH (pd:PatientData {patient_id: $patient_id}) DETACH DELETE pd", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'BoneDiseasePresent', value: 'false'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'ERStatus', value: 'negative'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'PRStatus', value: 'negative'})", patient_id=patient_id)
        session.run("CREATE (pd:PatientData {patient_id: $patient_id, data_type: 'HER2Status', value: 'positive'})", patient_id=patient_id)

    print("Test Case 6: BINV-20 with No Bone Disease, ER/PR-negative, HER2-positive")
    recommendation = loader.evaluate_patient(patient_id, "BINV-20")
    print(json.dumps(recommendation, indent=4))

    # Close connection
    loader.close()

    # Close connection
    loader.close()